# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)

# Show dataset metadata information
print(f"Dataset Title: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}\n")
print(f"License: {dataset.metadata.license}\n")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', '')}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their @id
record_sets = dataset.metadata.record_sets
print("Available RecordSets:")
for rs in record_sets:
    fields = getattr(rs, 'fields', [])
    field_ids = [field['@id'] if isinstance(field, dict) and '@id' in field else str(field) for field in fields]
    print(f"- RecordSet @id: {rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', rs)}")
    print(f"  Name: {getattr(rs, 'name', '')}")
    print(f"  Fields (@id): {field_ids}")
    print()

In [ ]:
# Show first few records from each RecordSet (@id references only)
for rs in record_sets:
    record_set_id = rs['@id'] if isinstance(rs, dict) else getattr(rs, '@id', rs)
    print(f"\nSample records for RecordSet @id: {record_set_id}")
    try:
        for i, rec in enumerate(dataset.records(record_set=record_set_id)):
            print(rec)
            if i >= 2:
                break
    except Exception as e:
        print(f"Could not fetch records: {e}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. All record set and field `@id`s used are as shown in the overview.

In [ ]:
# Build a dictionary of DataFrames for easy access, using RecordSet @ids
dataframes = {}
list_of_record_sets_ids = []

# Collect all @id's from record_sets
for rs in record_sets:
    if isinstance(rs, dict) and '@id' in rs:
        list_of_record_sets_ids.append(rs['@id'])
    elif hasattr(rs, '@id'):
        list_of_record_sets_ids.append(getattr(rs, '@id'))

for record_set_id in list_of_record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for RecordSet @id '{record_set_id}' with shape: {df.shape}")
        else:
            print(f"No records found for RecordSet @id '{record_set_id}'.")
    except Exception as e:
        print(f"Could not extract RecordSet @id '{record_set_id}': {e}")

# Examine columns (fields by @id) for the first available record set DataFrame
if dataframes:
    rs_id = list(dataframes.keys())[0]  # Use the first one for example
    print(f"\nFields for RecordSet @id '{rs_id}': ")
    print(dataframes[rs_id].columns.tolist())
    dataframes[rs_id].head()
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section can include steps like removing outliers, transforming distributions, or grouping data by key attributes.

In [ ]:
# For demonstration, pick a numeric field from the first available DataFrame

if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    
    # Try to determine a numeric field (@id) to use (fallback: pick first)
    numeric_field = None
    for col in df.columns:
        try:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field = col
                break
        except Exception:
            continue
    if not numeric_field:
        numeric_field = df.columns[0]  # fallback
    print(f"Using numeric field '@id': {numeric_field}\n")
    
    # Filtering
    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
    try:
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())
    except Exception as e:
        print(f"Error filtering by {numeric_field}: {e}")
        filtered_df = df
    
    # Normalization
    try:
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized" ]].head())
    except Exception as e:
        print(f"Error normalizing {numeric_field}: {e}")
    
    # Try grouping by another field (@id)
    group_field = None
    for col in df.columns:
        if col != numeric_field and df[col].nunique() < min(10, df.shape[0]//5):
            group_field = col
            break
    if group_field:
        try:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            print(grouped_df.head())
        except Exception as e:
            print(f"Error grouping by {group_field}: {e}")
    else:
        print("No suitable categorical field found for grouping.")
else:
    print("No dataframe for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes:
    df = list(dataframes.values())[0]
    if numeric_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field].dropna(), kde=True)
        plt.title(f"Distribution of {numeric_field}")
        plt.xlabel(numeric_field)
        plt.show()
        
    if group_field and group_field in df.columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and analyze the FAIR\^2 mass spectrometry dataset using the `mlcroissant` library.

- We dynamically loaded metadata and accessed all entities using their Croissant `@id` fields, ensuring unambiguous references.
- We listed available RecordSets and examined their fields.
- Tabular data for each RecordSet was loaded into Pandas DataFrames for further analysis.
- We carried out initial filtering, normalization, and grouping on numeric fields using only `@id`-based references.
- Basic visualizations were created for the distribution and groupwise comparison of numeric data.

Continue your exploration by referencing specific `@id`s from the Croissant schema to ensure full interoperability and reproducibility!